# Per-Frame Team-Classification Ground Truth: Labelling Tool

**Purpose:** An interactive labelling tool: a human labels one (frame, player) pair at
a time, in a fixed shuffled order across all three clips, through an ipywidgets click
UI.  
**Inputs:** `data/raw/{clip}.mp4` and the cached
`data/processed/{clip}/player_detections.pkl` tracks.  
**Outputs:** `data/annotations/team_assignment_gt_per_frame.csv`, appended after every
click, resumable.  
**Backs:** the shipped team ground truth in `data/annotations/`, which this tool
produced; the team-classification sweep is scored against it.

Labelling is per (frame, player) rather than one team per track for a whole clip:
labelling per track lets a human labeller remember a track's previous label and reuse
it across an ID switch, a human analogue of the sticky-cache problem this ground truth
exists to catch. All the sampling, persistence and resume logic lives in
`basketball/labelling/team_gt_sampling.py` (unit tested); this notebook only drives
the widgets.

**Kernel:** the JupyterHub Python kernel with `torch`/`ultralytics` installed (the
same kernel used for `training/` and `scripts/*.ipynb`); the video and cache files
this notebook reads exist there.

Progress is saved to `data/annotations/team_assignment_gt_per_frame.csv` after every
single click, so closing the kernel loses at most the frame in progress. Re-running
this notebook resumes automatically at the first unlabelled item.

## Setup

In [ ]:
import os
from pathlib import Path

# Cell paths below are all relative to the repo root, not wherever
# Jupyter happened to start.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
os.chdir(REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
from collections.abc import Callable

import ipywidgets as widgets
from IPython.display import display

from basketball.cache.cache_utils import load_cache
from basketball.utils.io_utils import load_video
from basketball.labelling.frame_rendering import crop_thumbnail, draw_tracked_boxes, encode_png
from basketball.labelling.team_gt_sampling import (
    CLIPS,
    CSV_PATH,
    PLAYER_DETECTIONS_CACHE_TEMPLATE,
    append_label,
    build_shuffled_order,
    default_frame_counts,
    default_gt_paths,
    frame_player_ids_from_tracks,
    load_existing_labels,
    load_labelled_rows,
    resume_index,
    sample_all_clips,
)

## Sample frames, build the fixed shuffled order, and resume

Frame counts come from each clip's cached player-track length (never a
hardcoded number). The sample per clip is the union of the every-5th-frame
backbone, clip_3's two occlusion windows, and the real MOT ground-truth
frames already labelled for tracking evaluation; see
`sample_clip_frame_indices`'s docstring for why the union guarantees every
MOT-GT frame is also a team-GT frame regardless of the MOT sampling stride.

In [ ]:
frame_counts = default_frame_counts()
gt_paths = default_gt_paths()
sample_frames = sample_all_clips(frame_counts, gt_paths)

for clip in CLIPS:
    print(f'{clip}: {frame_counts[clip]} cached frames -> {len(sample_frames[clip])} sampled')
print(f'TOTAL sampled frames: {sum(len(frames) for frames in sample_frames.values())}')

In [ ]:
# Cached player tracks, needed both for rendering (every visible player's
# box) and for resumability (which players exist on a given sampled frame).
tracks_by_clip = {clip: load_cache(PLAYER_DETECTIONS_CACHE_TEMPLATE.format(clip=clip)) for clip in CLIPS}

# Only the sampled frames are decoded and kept in memory, not the full clips.
frames_by_clip = {
    clip: {idx: frame for idx, frame in load_video(f'data/raw/{clip}.mp4') if idx in set(sample_frames[clip])}
    for clip in CLIPS
}
for clip in CLIPS:
    print(f'{clip}: {len(frames_by_clip[clip])} frames decoded and cached in memory')

In [ ]:
shuffled_order = build_shuffled_order(sample_frames)
frame_player_ids = frame_player_ids_from_tracks(sample_frames, tracks_by_clip)

labelled = load_existing_labels(CSV_PATH)
position = resume_index(shuffled_order, labelled, frame_player_ids)

print(f'{len(shuffled_order)} (clip, frame) items in the fixed shuffled order.')
print(f'{len(labelled)} (frame, player) labels already recorded at {CSV_PATH}.')
print(f'Resuming at position {position} of {len(shuffled_order)}.')

## Labelling UI

In [ ]:
TEAM_BUTTONS = [('Team 1', '1'), ('Team 2', '2'), ('Unclear', 'unclear')]
SELECTED_STYLE = 'success'
UNSELECTED_STYLE = ''

main_image = widgets.Image(format='png')
progress_label = widgets.Label()
player_rows_box = widgets.VBox()
prev_button = widgets.Button(description='Previous Frame', button_style='warning')
next_button = widgets.Button(description='Next Frame', button_style='info')
status_label = widgets.Label()

# player_id -> {team_value -> Button}, rebuilt every render() so a stale
# reference from the previous frame can never be clicked by accident.
player_buttons: dict[int, dict[str, widgets.Button]] = {}


def current_clip_frame() -> tuple[str, int]:
    clip, frame_idx = shuffled_order[position]
    return clip, frame_idx


def refresh_progress_label() -> None:
    total = len(shuffled_order)
    if position >= total:
        # resume_index()/on_next_clicked() can legitimately return an
        # out-of-range position once every sampled frame is fully
        # labelled -- "Frame N+1 of N" would be nonsense here.
        progress_label.value = f'All {total} sampled frames labelled — {len(labelled)} labels so far'
    else:
        progress_label.value = f'Frame {position + 1} of {total} — {len(labelled)} labels so far'


def highlight_selected(player_id: int, tracks: dict) -> None:
    """Set each of player_id's three buttons to reflect any already-recorded label for this (clip, frame, player)."""
    clip, frame_idx = current_clip_frame()
    buttons = player_buttons[player_id]
    for value, button in buttons.items():
        is_selected = (clip, frame_idx, player_id) in labelled and value == _recorded_team(clip, frame_idx, player_id)
        button.button_style = SELECTED_STYLE if is_selected else UNSELECTED_STYLE


def _recorded_team(clip: str, frame_idx: int, player_id: int) -> str | None:
    """The most recently written true_team for (clip, frame_idx, player_id), or None if never labelled."""
    # Re-reading the CSV on every click would be wasteful; label_lookup is
    # kept alongside `labelled` and updated in on_team_clicked() below. Both
    # are seeded from load_labelled_rows(), which is the single place a
    # correction (re-labelling the same key) is resolved to its latest
    # value -- this dict is never built from the raw, undeduplicated file.
    return label_lookup.get((clip, frame_idx, player_id))


label_lookup: dict[tuple, str] = {
    (row['clip'], int(row['frame_idx']), int(row['player_id'])): row['true_team']
    for row in load_labelled_rows(CSV_PATH)
}


def on_team_clicked(player_id: int, team_value: str, tracks: dict) -> Callable[[widgets.Button], None]:
    def handler(_button: widgets.Button) -> None:
        clip, frame_idx = current_clip_frame()
        append_label(clip, frame_idx, player_id, team_value, path=CSV_PATH)
        labelled.add((clip, frame_idx, player_id))
        label_lookup[(clip, frame_idx, player_id)] = team_value
        highlight_selected(player_id, tracks)
        refresh_progress_label()
    return handler


def render() -> None:
    if position >= len(shuffled_order):
        # Nothing left to label. resume_index() returns exactly this
        # out-of-range position when every sampled frame's players are
        # already fully labelled -- indexing shuffled_order[position]
        # below would raise IndexError; show a clear message instead.
        # on_next_clicked() below routes here too, so in-session completion
        # and startup completion go through this identical branch.
        main_image.value = b''
        player_buttons.clear()
        player_rows_box.children = []
        status_label.value = f'All {len(shuffled_order)} sampled frames are fully labelled — nothing left to label.'
        refresh_progress_label()
        return

    clip, frame_idx = current_clip_frame()
    frame = frames_by_clip[clip][frame_idx]
    tracks = tracks_by_clip[clip][frame_idx]

    annotated = draw_tracked_boxes(frame, tracks)
    main_image.value = encode_png(annotated)

    player_buttons.clear()
    rows = []
    for player_id, track in sorted(tracks.items()):
        thumbnail = widgets.Image(value=encode_png(crop_thumbnail(frame, track.bbox)), format='png', width=60)
        id_label = widgets.Label(value=f'ID {player_id}', layout=widgets.Layout(width='60px'))

        buttons = {}
        for text, value in TEAM_BUTTONS:
            button = widgets.Button(description=text, layout=widgets.Layout(width='90px'))
            button.on_click(on_team_clicked(player_id, value, tracks))
            buttons[value] = button
        player_buttons[player_id] = buttons

        # highlight_selected() runs after every button for this player
        # exists, so a frame revisited via Previous shows its existing
        # label (if any) pre-selected, exactly like a freshly rendered one.
        rows.append(widgets.HBox([id_label, thumbnail, *buttons.values()]))
        highlight_selected(player_id, tracks)

    player_rows_box.children = rows
    refresh_progress_label()
    status_label.value = f'{clip} frame {frame_idx}'


def on_prev_clicked(_button: widgets.Button) -> None:
    global position
    # Moves back exactly one place in shuffled_order, not to the previous
    # UNLABELLED item like Next does -- the labeller may specifically want
    # to revisit an already-labelled frame to correct a mis-click, and the
    # fixed shuffled order makes a specific earlier frame otherwise
    # unfindable once you've moved on. render() already displays whatever
    # frame `position` points to and pre-highlights any existing label, so
    # clicking a different team button here naturally produces the
    # correcting row through the existing append_label() -> dedup path.
    # No completion check needed here (unlike on_next_clicked below) --
    # moving back one place is well-defined regardless of what is or isn't
    # labelled elsewhere in shuffled_order.
    position = max(position - 1, 0)
    render()


def on_next_clicked(_button: widgets.Button) -> None:
    global position
    # Recomputed against the FULL shuffled_order every click, the same way
    # the startup resume above does it -- not the incremental
    # shuffled_order[position + 1:] slice this used to check. That slice
    # only ever looked AFTER the current position, which was correct back
    # when navigation was strictly forward-only, but Previous (above) makes
    # navigation non-monotonic: a labeller can go back, leave an earlier
    # frame incomplete, and continue, and a forward-only check can never
    # see that earlier gap. The same blind spot also fired under pure
    # forward navigation: leaving the LAST frame partially labelled and
    # clicking Next used to declare completion, since the slice after the
    # last position is always empty. A full scan over ~164 total frames
    # costs nothing; correctness matters far more here than the
    # micro-optimisation the slice was providing.
    position = resume_index(shuffled_order, labelled, frame_player_ids)
    render()


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

render()
display(widgets.VBox([status_label, main_image, player_rows_box, widgets.HBox([prev_button, next_button]), progress_label]))

## Post-hoc audit

Flags every labelled row whose `true_team` disagrees with its track's modal
team (excluding `unclear`): candidates for a genuine ID switch or a
labelling mistake, surfaced rather than silently resolved. A track with an
exact 50/50 split has no modal team at all (`track_modal_team` returns
`None` for it rather than picking a side by row order, since a perfect split is
precisely the ID-switch signature this audit exists to detect), in which
case every one of that track's rows is flagged. Cross-referenced against
`data/outputs/mot_evaluation/switches.csv` (from `scripts/run_evaluation.py`)
when it is present: `production_switch` reflects only the shipped
`'production'` configuration (the dissertation's cross-reference claim is
about that configuration specifically, not whichever of the five sweep
configurations happened to switch), while `matched_configs` separately
records every configuration, production or not, that recorded a switch
at that frame, for informational purposes.

In [ ]:
from basketball.labelling.team_gt_sampling import (
    cross_reference_switches,
    flag_disagreements,
    load_labelled_rows,
    load_switch_configs,
    load_switch_frames,
)

all_rows = load_labelled_rows(CSV_PATH)
flagged = flag_disagreements(all_rows)
switch_frames_by_clip = load_switch_frames()  # 'production' configuration only, by default
switch_configs_by_clip_frame = load_switch_configs()  # every configuration, for the informational matched_configs field
annotated_flags = cross_reference_switches(flagged, switch_frames_by_clip, switch_configs_by_clip_frame)

print(f'{len(all_rows)} total labels, {len(flagged)} disagree with (or have no) modal team.')
if not switch_frames_by_clip and not switch_configs_by_clip_frame:
    print('No data/outputs/mot_evaluation/switches.csv found — run scripts/run_evaluation.py '
          'first to cross-reference against known MOT ID switches.')

for row in annotated_flags:
    if row['production_switch']:
        coincidence = 'COINCIDES with a production-configuration switch'
    elif row['matched_configs']:
        matched = row['matched_configs']
        coincidence = f'no production switch, but matched non-production config(s) {matched}'
    else:
        coincidence = 'no known switch at this frame'
    clip = row['clip']
    frame_idx = row['frame_idx']
    player_id = row['player_id']
    true_team = row['true_team']
    print(f'{clip} frame {frame_idx} player {player_id}: labelled {true_team!r} ({coincidence})')